# AI-ML Assignment 8 — Handwritten Digit Recognition using Artificial Neural Networks (ANN)

**Total Marks:** 10

This notebook loads the MNIST handwritten digits dataset (from the Kaggle CSV files), preprocesses it, trains an ANN using TensorFlow/Keras, and evaluates its performance.

> **Setup:** Download `mnist_train.csv` and `mnist_test.csv` from [this Kaggle dataset](https://www.kaggle.com/datasets/oddrationale/mnist-in-csv) and place them in the same folder as this notebook before running.

## Task 1: Data Understanding (2 Marks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the dataset using Pandas
train_df = pd.read_csv("mnist_train.csv")
test_df = pd.read_csv("mnist_test.csv")

# Combine both files into one dataset; we will create our own 80/20 split in Task 2
df = pd.concat([train_df, test_df], ignore_index=True)

# 2. Display the first five records
df.head()

In [ ]:
# 3. Identify Input features and Target variable
# Target variable: 'label' (the digit, 0-9)
# Input features : the 784 pixel columns (1x1 ... 28x28), each a grayscale pixel value

target_variable = "label"
feature_columns = [c for c in df.columns if c != "label"]

print(f"Target variable: {target_variable}")
print(f"Number of input features (pixels): {len(feature_columns)}")

In [ ]:
# 4. Display the dataset dimensions and summary information
print("Dataset shape:", df.shape)
df["label"].value_counts().sort_index()

In [ ]:
# 5. Display one sample handwritten digit using Matplotlib
sample_index = 0
sample_image = df.loc[sample_index, feature_columns].values.reshape(28, 28)
sample_label = df.loc[sample_index, "label"]

plt.figure(figsize=(3, 3))
plt.imshow(sample_image, cmap="gray")
plt.title(f"Sample digit - Label: {sample_label}")
plt.axis("off")
plt.show()

## Task 2: Data Preprocessing (2 Marks)

In [ ]:
# Check for missing values
df.isnull().sum().sum()

In [ ]:
# Separate features and target variable
X = df[feature_columns].values
y = df["label"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Normalize pixel values to the range 0-1
X = X.astype("float32") / 255.0
print("Pixel value range after normalization:", X.min(), "to", X.max())

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

In [ ]:
from tensorflow.keras.utils import to_categorical

# Convert the target labels into categorical format using One-Hot Encoding
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print("y_train_cat shape:", y_train_cat.shape)
print("Example one-hot label:", y_train_cat[0], "-> digit", y_train[0])

## Task 3: Model Development (3 Marks)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# Build an Artificial Neural Network (ANN)
model = Sequential([
    Input(shape=(784,)),                 # Input Layer: one neuron per pixel
    Dense(128, activation="relu"),       # Hidden Layer 1: 128 neurons (ReLU)
    Dense(64, activation="relu"),        # Hidden Layer 2: 64 neurons (ReLU)
    Dense(10, activation="softmax"),     # Output Layer: 10 neurons (Softmax)
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Train the model for 10 epochs
history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=1,
)

In [ ]:
# Predict the handwritten digits on the test dataset
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

pd.DataFrame({"Actual": y_test[:10], "Predicted": y_pred[:10]})

## Task 4: Model Evaluation (2 Marks)

In [ ]:
# Test Accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Confusion Matrix
labels_present = sorted(np.unique(np.concatenate([y_test, y_pred])))
cm = confusion_matrix(y_test, y_pred, labels=labels_present)

plt.figure(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_present)
disp.plot(cmap="Blues", values_format="d", ax=plt.gca(), colorbar=True)
plt.title("Confusion Matrix - ANN Digit Classifier")
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Accuracy vs Epoch graph
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Loss vs Epoch graph
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

**Observations:**

1. Test accuracy is expected to be high (typically well above 95%) for this
   architecture on MNIST, since handwritten digits are a relatively clean,
   well-separated classification problem for a fully connected ANN.
2. The confusion matrix should show most errors concentrated between visually
   similar digit pairs (commonly 4/9, 3/5, or 7/1), rather than being spread evenly
   across all digit pairs, since these digits share similar stroke shapes.
3. The accuracy vs. epoch and loss vs. epoch curves should both show training and
   validation performance improving together over the 10 epochs; if validation loss
   starts rising while training loss keeps falling, that is a sign the model is
   beginning to overfit and would benefit from regularization or fewer epochs.
4. The classification report's per-digit precision/recall highlights whether any
   specific digit is systematically harder for the model — typically digits with
   more stroke variation (such as 8) tend to have slightly lower scores than more
   visually distinct digits (such as 0 or 1).

## Task 5: Conclusion (1 Mark)

This project built an Artificial Neural Network to classify handwritten digits from
the MNIST dataset, using two hidden layers of 128 and 64 ReLU neurons and a 10-neuron
softmax output layer. After normalizing pixel values to the 0–1 range and one-hot
encoding the labels, the model was trained for 10 epochs with the Adam optimizer and
categorical crossentropy loss, achieving strong test accuracy with most confusion
concentrated between visually similar digit pairs. Hidden layers are what give an ANN
its power: without them, the network could only learn a linear mapping from raw pixels
to digit classes, but the hidden layers let the model progressively combine simple
pixel patterns into more abstract, non-linear representations of stroke shapes and
digit structure, which is essential for separating classes that are not linearly
separable in raw pixel space. Compared to traditional machine learning approaches such
as KNN or a single Decision Tree, one advantage of this deep learning approach is that
it automatically learns useful feature representations directly from raw pixel data,
rather than requiring manual feature engineering. A limitation of ANNs, however, is
that a fully connected network like this one treats each pixel independently and
ignores the 2D spatial structure of the image, meaning it cannot recognize shifted,
rotated, or scaled versions of a digit as easily as an architecture designed for
images (such as a Convolutional Neural Network) would.